In [1]:
import pandas as pd
from deepeval.metrics import GEval
from deepeval.models import GPTModel
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from dotenv import load_dotenv
import warnings
import os

/tmp/ipykernel_1401/3988472673.py:4: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [2]:
load_dotenv()

API_KEY = os.getenv('API_KEY')

In [3]:
warnings.filterwarnings('ignore')
os.environ["CONFIDENT_METRIC_LOGGING_VERBOSE"] = "0"

In [4]:
abstracts = pd.read_csv('/home/vinicius/Área de Trabalho/Artigos/llm_artigo/llmArtigo/datasets/Abstracts_ORA - abstracts.csv')


In [5]:
abstracts = abstracts.drop(['division', 'department'], axis=1)

In [6]:
abstracts

,title,abstract
0,Niu_2025_Functional_Nanolayer_Dielectrics,Negatively charged dielectric films have signi...
1,Brody_2025_Numerical_simulation_of,Hypersonic flight is an essential component of...
2,Ehrenfels_2025_The_epistemology_of,This thesis addresses the question ‘Under what...
3,Greenrod_2025_Temperature_as_a,Thermal change has a profound impact on specie...
4,Chao_2025_Synthetic_transmembrane_transporters,The thesis describes the design and synthesis ...
5,Gao_2025_Association_between_neuroticism,"BACKGROUND: Neuroticism, a personality trait r..."
6,Parkes_2025_Magnetically-activated_DNA_and,Nucleic acids serve as the fundamental buildin...
7,Taskesen_2025_Tuning_magnetism_and,This thesis reports the synthesis and characte...
8,Rad_2025_Police_Unionism,"In 2024, there were over 1,300 incidents of po..."
9,Zong_2025_Sustainable_removal_of,Conventional wastewater treatment often fails ...


In [9]:
df = pd.read_csv('/home/vinicius/Área de Trabalho/Artigos/llm_artigo/llmArtigo/datasets/lead_800.csv')
df

,title,leadn_response
0,Niu_2025_Functional_Nanolayer_Dielectrics,Chapter 1 Introduction 1.1 Semiconductor Devic...
1,Brody_2025_Numerical_simulation_of,Chapter 1 Introduction “[T]he hypersonic regim...
2,Ehrenfels_2025_The_epistemology_of,"1 Introduction Remarkably, in science we often..."
3,Greenrod_2025_Temperature_as_a,Introduction Section 1.3 has been published in...
4,Chao_2025_Synthetic_transmembrane_transporters,Chapter 1 - 1 - Chapter 1 Introduction Chapter...
5,Gao_2025_Association_between_neuroticism,Chapter 1 Introduction 17 1.1 DEMENTIA Definit...
6,Parkes_2025_Magnetically-activated_DNA_and,1. Introduction 1.1. Structure and function of...
7,Taskesen_2025_Tuning_magnetism_and,Chapter 1. Introduction 1.1 Solid-state Chemis...
8,Rad_2025_Police_Unionism,"CHAPTER 1: INTRODUCTION In 2024, law enforceme..."
9,Zong_2025_Sustainable_removal_of,Chapter 1: Introduction 1.1 Background The gro...


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   title           30 non-null     str  
 1   leadn_response  30 non-null     str  
dtypes: str(2)
memory usage: 612.0 bytes


In [11]:
model = GPTModel(
    model = "gpt-5",
    temperature= 0,
    api_key= API_KEY
)

In [12]:
model_columns = [col for col in df.columns if col.endswith("_response")]
model_columns

['leadn_response']

In [13]:
metric = GEval(
    name="Abstract_Quality",
    model= model,
    evaluation_steps=[
        "Verify whether the generated abstract preserves the core scientific elements of the reference, including problem, objectives, methodology, results, and contributions.",
    
        "Assess semantic fidelity: check if the generated abstract accurately represents the meaning of the reference without introducing distortions or hallucinations.",
    
        "Check for missing critical information from the reference abstract and penalize significant omissions.",
    
        "Ensure the output follows the style of a scientific abstract, without bullet points, markdown formatting, reasoning traces, or meta-commentary.",
    
        "Evaluate conciseness: the abstract should be compact and avoid unnecessary verbosity or redundancy."
        ],
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],

)

In [15]:
import pandas as pd
import os
from deepeval.test_case import LLMTestCase
from deepeval import evaluate

output_file = "geval_results_leadn.csv"

for idx in range(len(df)):
    title = df.loc[idx, "title"] if "title" in df.columns else f"thesis_{idx}"
    reference_abstract = abstracts.loc[idx, "abstract"]

    for model_col in model_columns:
        model_name = model_col.replace("_response", "")
        candidate_summary = df.loc[idx, model_col]

        # Pula inferências nulas ou vazias
        if pd.isna(candidate_summary) or str(candidate_summary).strip() == "":
            print(f"[SKIP] idx={idx} | model={model_name} → inferência vazia")
            result_row = {
                "idx":          idx,
                "title":        title,
                "model":        model_name,
                "geval_score":  None,
                "geval_reason": "skipped: empty inference"
            }
        else:
            try:
                test_case = LLMTestCase(
                    input=title,
                    actual_output=str(candidate_summary),
                    expected_output=str(reference_abstract)
                )
                metric.measure(test_case)
                result_row = {
                    "idx":          idx,
                    "title":        title,
                    "model":        model_name,
                    "geval_score":  metric.score,
                    "geval_reason": metric.reason
                }
                print(f"[OK] idx={idx} | model={model_name} | score={metric.score:.3f}")

            except Exception as e:
                print(f"[ERROR] idx={idx} | model={model_name} → {e}")
                result_row = {
                    "idx":          idx,
                    "title":        title,
                    "model":        model_name,
                    "geval_score":  None,
                    "geval_reason": f"error: {str(e)}"
                }

        # Salva incrementalmente — protege contra crashes
        temp_df = pd.DataFrame([result_row])
        if os.path.exists(output_file):
            temp_df.to_csv(output_file, mode="a", header=False, index=False)
        else:
            temp_df.to_csv(output_file, mode="w", header=True, index=False)

print(f"\n✅ Avaliação concluída! Resultados salvos em '{output_file}'")

[OK] idx=0 | model=leadn | score=0.100


[OK] idx=1 | model=leadn | score=0.200


[OK] idx=2 | model=leadn | score=0.200


[OK] idx=3 | model=leadn | score=0.100


[OK] idx=4 | model=leadn | score=0.000


...

[OK] idx=25 | model=leadn | score=0.100


[OK] idx=26 | model=leadn | score=0.000


[OK] idx=27 | model=leadn | score=0.000


[OK] idx=28 | model=leadn | score=0.100


[OK] idx=29 | model=leadn | score=0.200

✅ Avaliação concluída! Resultados salvos em 'geval_results_leadn.csv'
